In [ ]:
# ===========================================================================
# Cell P0: setup and pools.
#
# H and L are the same rows. label_real.to_instances returns {**r, "label": llm_label,
# "gold_label": r["label"]}, so both arms share sent_id, text, markers and pair set, and
# H is recovered by swapping the label back. That is the construction behind the 0.785
# against 0.657 in meeting_results.md, and building H off train_f instead would quietly
# include pairs L does not have.
#
# The consequence for this stage: build_paraphrase_specs reads sent_id, text and spans
# and never touches label, so its output is identical for the two pools. The verifier
# judges what a sentence asserts, not what a label claims, so one verdict frame answers
# both arms. Paraphrase once, verify once, score twice.
#
#   paraphrase   ~2,400 sentences x 3 = ~7,200 calls, low effort
#   verify       ~7,200 calls, high effort
#
# reasoning_effort on the verifier stays at its default of "high". paraphrase_plan.md
# assumed low and that is wrong: at low the five-way verifier's NONE recall collapsed to
# 0.45-0.48, and the 0.920 NONE recall and 0.948 positive recall this whole diagnostic
# rests on are high-effort numbers. The v18 full verify ran at max_workers=8, so check
# whether that was a rate limit or a habit before assuming this fits in twenty minutes.

In [ ]:
# ===========================================================================
import os
import re
import json
import random
import importlib
from collections import Counter, defaultdict

import pandas as pd
from openai import OpenAI

import ddi.paraphrase, ddi.verify_binary, ddi.pruning, ddi.label_real, ddi.synth
for m in (ddi.paraphrase, ddi.verify_binary, ddi.pruning, ddi.label_real, ddi.synth):
    importlib.reload(m)

from ddi import label_real
from ddi.data import build_human, render_pair
from ddi.manifest import load_dataset
from ddi.synth import generate_raw, RAW, SynthEntity
from ddi.verify_binary import (build_batches, make_binary_verifier, load_verdicts,
                               calibration_report, group_by_sentence,
                               _sentence_spans, _marked_spans, _strip, _number_inline)
from ddi.pruning import instance_keys
from ddi.paraphrase import NUMBERED, build_paraphrase_specs, make_paraphraser

MODEL, API = "gpt-oss-120b", "responses"
LABEL_VERDICTS = "datasets/instances/raw/label-high-c8.jsonl"

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=120.0)

train, dev, val = build_human()

label_inst, dropped = label_real.to_instances(train, LABEL_VERDICTS)
POOLS = {
    "L": label_inst,
    "H": [{**r, "label": r["gold_label"]} for r in label_inst],
}
print(f"labelling arm: {len(label_inst)} instances, "
      f"{len({r['sent_id'] for r in label_inst})} sentences, "
      f"{dropped} dropped for shape mismatch")
for n, inst in POOLS.items():
    pos = sum(1 for r in inst if r["label"] != "NONE") / len(inst)
    print(f"{n}: {len(inst)} pairs, {len(group_by_sentence(inst))} sentences, "
          f"pos rate {pos:.3f}")
print("corpus pair positive rate 0.162; L is expected near 0.242")

# the two pools must be row-for-row identical apart from the label
assert [r["sent_id"] for r in POOLS["H"]] == [r["sent_id"] for r in POOLS["L"]]
assert [r["text"] for r in POOLS["H"]] == [r["text"] for r in POOLS["L"]]

BASE_POOL = POOLS["H"]        # either works; spans and text are shared

In [ ]:
# ===========================================================================
# Cell P1: pilot, 200 sentences. 600 calls, a few minutes.
#
# The point is to find out whether the pass is inert before paying for the rest.
# build_paraphrase_specs applies its own skip_pairs_above=190, so the enumeration
# sentences are already excluded and no train_f-style filter is needed here.

In [ ]:
# ===========================================================================
paraphrase = make_paraphraser(client, model=MODEL, reasoning_effort="low", api=API)

SPECS = build_paraphrase_specs(BASE_POOL, k=3)
sent_ids = sorted({s["sent_id"] for s in SPECS})
PILOT = [s for s in SPECS if s["sent_id"] in set(sent_ids[:200])]
print(f"{len(SPECS)} specs over {len(sent_ids)} sentences, {len(PILOT)} in pilot")

generate_raw(PILOT, paraphrase, gen_id="para-pilot", max_workers=32)

In [ ]:
# ===========================================================================
# Cell P2: the inertness gate.
#
# paraphrase_report counts survivors; it does not measure whether anything changed, which
# is the abort condition its own docstring names. Two numbers:
#
#   token Jaccard  original against rewrite, tags stripped. High median means the model
#                  reworded nothing and the agreement signal measures nothing.
#   distinct-4     v18 sits at 0.896 against the corpus's 0.973.
#
# The thresholds below are judgement, not measurement, which is the defect the week 6
# gates had. Calibrate them first: reword three corpus sentences by hand, push them
# through _tokens, and see where a real paraphrase lands. Five minutes, and it turns a
# guess into a reference point. Do not skip it and then quote the thresholds.

In [ ]:
# ===========================================================================
def _tokens(s):
    return [t for t in re.sub(r"\[\d+\]", "", s).lower().split() if t.isalnum()]


def distinct_n(texts, n=4):
    g = [tuple(t[i:i + n]) for t in map(_tokens, texts)
         for i in range(max(0, len(t) - n + 1))]
    return len(set(g)) / max(len(g), 1)


def read_raw(gen_id):
    """Successful records only, as (spec, sample) pairs."""
    out = []
    for line in (RAW / f"{gen_id}.jsonl").read_text().splitlines():
        if not line:
            continue
        r = json.loads(line)
        if r.get("error") or not r.get("sample"):
            continue
        out.append((r["spec"], r["sample"]))
    return out


def divergence_report(gen_id, specs):
    orig = {s["sent_id"]: s["numbered"] for s in specs}
    got = read_raw(gen_id)

    jac, rewrites = [], []
    for _, s in got:
        o = orig.get(s["sent_id"])
        if not o:
            continue
        a, b = set(_tokens(o)), set(_tokens(s["numbered"]))
        jac.append(len(a & b) / max(len(a | b), 1))
        rewrites.append(s["numbered"])
    jac.sort()
    med = jac[len(jac) // 2] if jac else 1.0
    d0, d1 = distinct_n(list(orig.values())), distinct_n(rewrites)

    print(f"{len(got)} rewrites kept of {len(specs)} specs")
    print(f"token Jaccard  p10 {jac[len(jac)//10]:.3f}  median {med:.3f}  "
          f"p90 {jac[9*len(jac)//10]:.3f}")
    print(f"distinct-4  original {d0:.3f} -> rewrites {d1:.3f}  ({d1 - d0:+.3f})")
    print(f"{'INERT, do not proceed' if med > 0.80 or d1 - d0 < 0.01 else 'proceed'}")


divergence_report("para-pilot", PILOT)


# ---------------------------------------------------------------------------
# Tag preservation by mention count. make_paraphraser raises when the tag set changed and
# generate_raw records that as an error, so if failures concentrate in dense sentences the
# surviving pool is biased toward short simple ones and every downstream number inherits
# it. The labelling arm already failed this way.
# ---------------------------------------------------------------------------
def failure_by_mentions(gen_id):
    tot, fail = Counter(), Counter()
    for line in (RAW / f"{gen_id}.jsonl").read_text().splitlines():
        if not line:
            continue
        r = json.loads(line)
        n = r["spec"]["n_mentions"]
        tot[n] += 1
        if r.get("error") or not r.get("sample"):
            fail[n] += 1
    print(f"{'mentions':>9} {'total':>7} {'failed':>7} {'rate':>7}")
    for n in sorted(tot):
        print(f"{n:>9} {tot[n]:>7} {fail[n]:>7} {fail[n] / tot[n]:>7.3f}")


failure_by_mentions("para-pilot")
print("""
  rate flat in mention count      -> the surviving pool is unbiased, proceed
  rate rising with mention count  -> dense sentences are being lost. Drop k to 2 and
     raise max_output_tokens, or accept it and say so; do not report a pruning number
     from a pool that quietly excludes the hard sentences.
""")

In [ ]:
# ===========================================================================
# Cell P3: full pass. Unattended, tmux. resume=True is already the default, and the
# pilot's successes are in a different gen_id so nothing is reused; that is deliberate,
# the pilot is disposable.

In [ ]:
# ===========================================================================
generate_raw(SPECS, paraphrase, gen_id="para", max_workers=64)

In [ ]:
# ===========================================================================
# Cell P4: verifier specs for the paraphrases. No span arithmetic.
#
# build_paraphrase_specs numbers with _number_inline(_strip(text), _sentence_spans(rows)),
# the identical call build_batches makes, so the tags in a rewrite are the tags
# build_batches gave the original. The model must carry them through verbatim and
# make_paraphraser rejects the rewrite otherwise.
#
# So a paraphrase verifier spec is the original's build_batches spec with `text` swapped
# for the rewrite. pairs, names and n_mentions carry over unchanged, load_verdicts returns
# m1/m2 as those same tags, and agreement keys on (sent_id, m1, m2) across the original
# and all three rewrites with nothing to align.
#
# `gold` is taken from H, so VDF.gold_pos is human gold. Use it for calibration only.
# Never read gold_pos when scoring L; L's labels come from the pool, not from this frame.

In [ ]:
# ===========================================================================
def para_verify_specs(gen_id):
    base = {b["sent_id"]: b for b in build_batches(POOLS["H"])}
    out, missing = [], 0
    for _, s in read_raw(gen_id):
        b = base.get(s["sent_id"])
        if b is None:
            missing += 1
            continue
        out.append({**b, "sent_id": f"{s['sent_id']}#p{s['para']}",
                    "text": s["numbered"]})
    print(f"{len(out)} paraphrase specs, {missing} with no original batch")
    return out


PARA_SPECS = para_verify_specs("para")

# reasoning_effort left at its default of "high" deliberately, see cell P0.
verifier = make_binary_verifier(client, model=MODEL, api=API)
generate_raw(PARA_SPECS, verifier, gen_id="para-verify", max_workers=8)

VDF = load_verdicts("para-verify")
print(f"{len(VDF)} verdict rows over {VDF.sent_id.nunique()} paraphrases")


# ---------------------------------------------------------------------------
# The instrument on rewritten text, scored against human gold. It was calibrated at 0.920
# NONE recall and 0.948 positive recall on original corpus text. If it lands far off that
# here, the rewrites have made the task harder and the disagreement rates in P5 are
# measuring that rather than label fragility. This is the check that decides whether the
# H-versus-L comparison means anything.
# ---------------------------------------------------------------------------
calibration_report(VDF)

In [ ]:
# ===========================================================================
# Cell P5: agreement, on the binary axis, scored twice off one verdict frame.
#
# Binary, not five-way. agreement_prune's docstring assumes labels, but the only
# calibrated instrument in the repo is binary. A five-way labeller means using the
# annotator, whose entire known deficit is precision, and precision is the axis this is
# meant to measure, so it would confound the result with the instrument. Binary is also
# the axis that matters: L over-predicts positives by 41%, precision 0.533 against 0.749.
#
# agreement_prune and apply_prune from paraphrase.py are not used. agreement_prune parses
# a jsonl payload with a `labels` list nothing produces, and apply_prune indexes
# positionally into group_by_sentence rows, which diverges from build_batches ordering on
# any sentence with nested spans, since build_batches skips those. instance_keys is the
# supported mapping and is what pruning.align uses.

In [ ]:
# ===========================================================================
PARA_SUFFIX = re.compile(r"#p\d+$")

VOTES = defaultdict(list)
for v in VDF.itertuples():
    VOTES[(PARA_SUFFIX.sub("", v.sent_id), v.m1, v.m2)].append(bool(v.flagged))
print(f"{len(VOTES)} pairs with at least one paraphrase verdict")
print(f"vote counts: {Counter(len(v) for v in VOTES.values()).most_common()}")


def agreement(name, mode="unanimous"):
    pool = POOLS[name]
    keys, unmappable = instance_keys(pool)
    by_sent = group_by_sentence(pool)
    ordered = [r for sid in by_sent for r in by_sent[sid]]

    keep, drop, novote = set(), set(), 0
    by_label = defaultdict(lambda: [0, 0])
    for r, k in zip(ordered, keys):
        if k is None:
            continue
        vs = VOTES.get(k)
        if not vs:
            novote += 1
            continue
        orig = r["label"] != "NONE"
        agree = sum(1 for v in vs if v == orig)
        ok = agree == len(vs) if mode == "unanimous" else agree * 2 > len(vs)
        (keep if ok else drop).add(k)
        by_label["positive" if orig else "NONE"][0 if ok else 1] += 1

    rate = len(drop) / max(len(keep) + len(drop), 1)
    print(f"{name} {mode}: keep {len(keep)}, drop {len(drop)}, prune rate {rate:.3f}, "
          f"{novote} pairs with no verdict, {unmappable} unmappable spans")
    for lab, (k, d) in by_label.items():
        print(f"    {lab:<9} kept {k:>6} dropped {d:>6} rate {d / max(k + d, 1):.3f}")
    return keep, rate


keep_H, rate_H = agreement("H", "unanimous")
keep_L, rate_L = agreement("L", "unanimous")
keep_L_maj, _ = agreement("L", "majority")

print(f"""
  H prune rate {rate_H:.3f}, L prune rate {rate_L:.3f}

  Same sentences, same rewrites, same verdicts. The only thing separating these two
  numbers is which label each pair started with, which is exactly what the diagnostic is
  supposed to isolate.

  L clearly above H  -> the instrument is measuring label fragility and the pruning arms
     are worth training. Expected, since L's deficit is precision and unstable positives
     are what this drops.
  H close to L       -> gold disagrees with itself under rewording at the same rate, so
     this is paraphrase noise and not label fragility. Drop L+A and L+P+A from stage 1
     and report the calibration as a negative result. Cheap, and worth having.
  H above L          -> the instrument is broken rather than informative. Stop and read
     twenty H disagreements before doing anything else.
""")


# ---------------------------------------------------------------------------
# Read ten of them either way. The verifier's false-positive rate on human dev is 8.0%,
# so a prune rate near that floor is noise and only the reading distinguishes the two.
# Both labels are printed because H and L share the row.
# ---------------------------------------------------------------------------
row_of = {}
for r, k in zip([r for sid in group_by_sentence(POOLS["L"]) for r in
                 group_by_sentence(POOLS["L"])[sid]], instance_keys(POOLS["L"])[0]):
    if k:
        row_of[k] = r

text_of = {sid: _number_inline(_strip(rows[0]["text"]), _sentence_spans(rows))
           for sid, rows in group_by_sentence(POOLS["L"]).items()}
para_of = defaultdict(list)
for _, s in read_raw("para"):
    para_of[s["sent_id"]].append(s["numbered"])

drop_L = [k for k in row_of if k not in keep_L]
for k in random.Random(0).sample(drop_L, min(10, len(drop_L))):
    sid, m1, m2 = k
    r = row_of[k]
    print(f"\n{'-' * 70}\nmentions {m1} and {m2}   "
          f"llm={r['label']}  gold={r['gold_label']}  votes={VOTES[k]}")
    print(f"  orig: {text_of.get(sid, '?')}")
    for p in para_of.get(sid, [])[:3]:
        print(f"  para: {p}")

In [ ]:
# ===========================================================================
# Cell P6: paraphrases as trainable instances, for the +P arms.
#
# The diagnostic above needed no spans. Training on the rewrites does, because train.py
# reads [E1]/[E2] markers. render_pair does the insertion and SynthEntity supplies the
# .locations interface it expects, so only the tag-position scan is hand-rolled.
#
# Built once and labelled twice, since H and L share the rows.
#
# CHECK the printout before training on 14,000 of them: the marked strings must be the
# drug names. An off-by-one here is silent and would move every +P number.

In [ ]:
# ===========================================================================
def para_instances(gen_id):
    base = {b["sent_id"]: b for b in build_batches(POOLS["H"])}
    keys, _ = instance_keys(POOLS["L"])
    by_sent = group_by_sentence(POOLS["L"])
    src_of = {k: r for r, k in
              zip([r for sid in by_sent for r in by_sent[sid]], keys) if k}

    out, skipped = [], Counter()
    for _, s in read_raw(gen_id):
        sid = s["sent_id"]
        b = base.get(sid)
        if b is None:
            skipped["no original"] += 1
            continue

        plain, pos, cursor = [], {}, 0
        num = s["numbered"]
        for m in NUMBERED.finditer(num):
            plain.append(num[cursor:m.start()])
            start = sum(len(p) for p in plain)
            plain.append(m.group(1))
            pos[int(m.group(2))] = (start, start + len(m.group(1)))
            cursor = m.end()
        plain.append(num[cursor:])
        text = "".join(plain)

        if sorted(pos) != list(range(1, b["n_mentions"] + 1)):
            skipped["tag set mismatch"] += 1
            continue

        for m1, m2 in b["pairs"]:
            src = src_of.get((sid, m1, m2))
            if src is None:
                skipped["no source row"] += 1
                continue
            out.append({**src,
                        "sent_id": f"{sid}#p{s['para']}",
                        "text": render_pair(text, SynthEntity(None, *pos[m1]),
                                            SynthEntity(None, *pos[m2]), mode="markers")})
    print(f"{len(out)} paraphrase instances, skipped {dict(skipped)}")
    return out


PARA_L = para_instances("para")                                    # carries LLM labels
PARA_H = [{**r, "label": r["gold_label"]} for r in PARA_L]          # and gold

for r in PARA_L[:10]:
    plain = _strip(r["text"])
    print(f"{r['label']:<10} {[plain[b:e] for b, e in _marked_spans(r['text'])]}")

In [ ]:
# ===========================================================================
# Cell P7: stage 1 arms, on val.
#
# val, not dev. Every number to date is dev and dev has been iterated against since
# week 2. This grid is a selection procedure, which is what val is for.
#
# Positive rate is printed for every arm deliberately. L runs 0.242 against the corpus's
# 0.162, and any arm that moves it toward 0.162 could be winning on class balance rather
# than on content. Note which arms move it before reading the F1 column.

In [ ]:
# ===========================================================================
from ddi.train import train_and_eval
from ddi.experiment import log_run

BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
        "neg_ratio": None, "render_mode": "markers"}


def train_arm(name, records, seeds=(0, 1, 2), notes=""):
    rows = []
    for seed in seeds:
        cfg = {**BASE, "seed": seed, "dataset": name}
        m = train_and_eval(cfg, records, val)
        log_run(cfg, m, notes=notes or name)
        rows.append({"arm": name, "seed": seed, "f1": m["micro_f1_pos"],
                     "p": m["micro_p_pos"], "r": m["micro_r_pos"]})
        print(f"{name} seed={seed} f1={m['micro_f1_pos']:.3f} "
              f"p={m['micro_p_pos']:.3f} r={m['micro_r_pos']:.3f}")
    df = pd.DataFrame(rows)
    print(f"{name}: f1 {df.f1.mean():.3f} (sd {df.f1.std():.3f})  "
          f"P {df.p.mean():.3f}  R {df.r.mean():.3f}  n={len(df)}")
    return df


def keep_only(instances, keep):
    keys, _ = instance_keys(instances)
    by_sent = group_by_sentence(instances)
    ordered = [r for sid in by_sent for r in by_sent[sid]]
    return [r for r, k in zip(ordered, keys) if k is not None and k in keep]


ARMS = {
    "H":       POOLS["H"],
    "H+P":     POOLS["H"] + PARA_H,
    "L":       POOLS["L"],
    "L+P":     POOLS["L"] + PARA_L,
    "L+A":     keep_only(POOLS["L"], keep_L),
    "L+A-maj": keep_only(POOLS["L"], keep_L_maj),
    "L+P+A":   keep_only(POOLS["L"], keep_L) + PARA_L,
}

for name, recs in ARMS.items():
    pos = sum(1 for r in recs if r["label"] != "NONE") / len(recs)
    print(f"{name:<10} {len(group_by_sentence(recs)):>6} sentences  "
          f"{len(recs):>7} pairs  pos rate {pos:.3f}")

res = [train_arm(n, r, notes="stage 1, paraphrase and agreement")
       for n, r in ARMS.items()]
summary = pd.concat(res)
print(summary.groupby("arm")[["f1", "p", "r"]].agg(["mean", "std", "count"]).to_string())
print("""
reference, dev, so not directly comparable to these val numbers:
  human labels        0.785   P 0.749  R 0.824   sd 0.009
  LLM labels          0.657   P 0.533  R 0.857   sd 0.008
  synthetic v18 low   0.462   P 0.349  R 0.689   sd 0.006

  L+A gains precision, recall roughly flat  -> the intervention did what it was aimed at
  L+A gains F1 with pos rate moved to 0.162 -> may be class balance, and the positive-rate
     control in stage 2 is the only thing that separates them. Do not claim content
     without it.
  L+P gains and L+A does not                -> augmentation works, the fragility
     diagnostic does not. Separate findings, report them separately.
  H+P above H                               -> paraphrase is worth something even on gold
     text. Strongest result available here and the most surprising, so re-check the P6
     span rendering before believing it.
""")